In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os


def load_results(test_function, kernel, strategy):
    save_dir = f"results/{test_function}/{strategy}/{kernel}"
    ys = []
    ts = []
    for filename in sorted(os.listdir(save_dir)):
        results = np.load(f"{save_dir}/{filename}")
        ys.append(results["ymin"])
        ts.append(results["time"])
    return np.array(ys), np.array(ts)


for test_function in ["goldstein", "lunar", "push", "rover"]:
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    for i, (strategy, color, kernel, linestyle) in enumerate(
        [
            ("lhs", "#1f77b4", "matern52", "solid"),
            ("lhs", "#1f77b4", "squaredexponential", "dashed"),
            ("bfgs", "#ff7f0e", "matern52", "solid"),
            ("bfgs", "#ff7f0e", "squaredexponential", "dashed"),
            ("voronoi", "#2ca02c", "matern52", "solid"),
            ("voronoi", "#2ca02c", "squaredexponential", "dashed"),
        ]
    ):
        y, t = load_results(test_function, kernel, strategy)
        if y.size == 0:
            continue
        runs, max_acquisitions = y.shape
        x = np.arange(max_acquisitions)

        # ci on median
        qu = np.clip(0.5 + 1.96 * np.sqrt(0.25 / runs), 0, 1)
        ql = np.clip(0.5 - 1.96 * np.sqrt(0.25 / runs), 0, 1)
        ul = np.quantile(y, qu, axis=0)
        ll = np.quantile(y, ql, axis=0)

        plt.plot(
            x,
            np.median(y, axis=0),
            label=f"{kernel} + {strategy} ({runs} runs)",
            color=color,
            linestyle=linestyle,
        )
        plt.fill_between(x, ll, ul, alpha=0.2, color=color)
    plt.title(f"{test_function.capitalize()}")
    plt.xlabel("Number of Acquisitions")
    plt.ylabel("Minimum Function Value")
    plt.legend()
    plt.grid()
    
    plt.subplot(1, 2, 2)
    for i, (strategy, kernel) in enumerate(
        [
            ("lhs", "matern52"),
            ("lhs", "squaredexponential"),
            ("bfgs", "matern52"),
            ("bfgs", "squaredexponential"),
            ("voronoi", "matern52"),
            ("voronoi", "squaredexponential"),
        ]
    ):
        y, t = load_results(test_function, kernel, strategy)
        if y.size == 0:
            continue
        plt.boxplot(t[:], positions=[i], patch_artist=True)
    plt.xticks(range(6), ["LHS + Matern52", "LHS + SE", "BFGS + Matern52", "BFGS + SE", "Voronoi + Matern52", "Voronoi + SE"])
    plt.ylabel("Time (seconds)")
    plt.ylim(0, None)
    plt.grid(True)
    plt.show()